In [1]:
# ============================================================
# 1. ENVIRONMENT
# ============================================================

!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "datasets>=2.20,<3" \
    "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.3" "tqdm>=4.65"

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
from transformers import pipeline

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python environment ready.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 63.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which

In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/phase4_4_5_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Where to look for the Phase 4.3 output files you uploaded.
# The loader below checks all of these, in order, and also globs
# /kaggle/input/*/ so you don't have to hardcode a dataset slug.
SEARCH_DIRS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("/kaggle/working/phase4_3_outputs"),
    Path("."),
]

PHASE_4_3_FILES = {
    "summary": "phase4_3_summary.csv",
    "entities": "phase4_3_entities.csv",
    "structured": "phase4_3_structured_incidents.jsonl",
    "gold_template": "gold_annotation_template.csv",
}

# Dataset + model, same as Phase 4.3.
HF_DATASET_ID = "QCRI/HumAID-all"
NER_MODEL_NAME = "dslim/bert-base-NER"

# Which splits to include in the FINAL Phase 4.5 export.
# None = use every split the dataset actually reports.
FINAL_SPLITS = None

# Cap rows per split for a quick top-to-bottom test run.
# Set to None for the real final export (full corpus).
ROW_LIMIT_PER_SPLIT = 300

GENERAL_NER_THRESHOLD = 0.55
RULE_HIGH = 0.92
RULE_MEDIUM = 0.85
RULE_LOW = 0.75

GOLD_FILE_NAME = "gold_annotations.csv"

print("Output directory:", OUTPUT_DIR)
print("ROW_LIMIT_PER_SPLIT:", ROW_LIMIT_PER_SPLIT, "(set to None for the real final run)")

Output directory: /kaggle/working/phase4_4_5_outputs
ROW_LIMIT_PER_SPLIT: 300 (set to None for the real final run)


In [3]:
# ============================================================
# 3. LOCATE PHASE 4.3 OUTPUTS
# ============================================================

def find_input_file(filename, search_dirs):
    '''Search fixed directories plus every /kaggle/input/*/ subfolder.'''
    candidates = []

    for d in search_dirs:
        if d.name == "input" and d.exists():
            candidates.extend(d.glob(f"*/{filename}"))
            candidates.extend(d.glob(filename))
        elif d.exists():
            candidates.append(d / filename)

    for c in candidates:
        if c.exists():
            return c

    return None

FOUND_PATHS = {
    key: find_input_file(fname, SEARCH_DIRS)
    for key, fname in PHASE_4_3_FILES.items()
}

print("Phase 4.3 files located:")
for key, path in FOUND_PATHS.items():
    print(f"  {key:14s} -> {path if path else 'NOT FOUND'}")

if FOUND_PATHS["gold_template"] is None:
    print(
        "\nWarning: gold_annotation_template.csv not found. "
        "Phase 4.4 annotation-assist steps will be skipped until it's available."
    )

Phase 4.3 files located:
  summary        -> NOT FOUND
  entities       -> NOT FOUND
  structured     -> NOT FOUND
  gold_template  -> NOT FOUND



In [4]:
# ============================================================
# 4. AUDIT
# ============================================================

audit_entities_df = None
audit_structured_records = None

if FOUND_PATHS["entities"] is not None:
    audit_entities_df = pd.read_csv(FOUND_PATHS["entities"])

    short_general = audit_entities_df[
        (audit_entities_df["source"] == "general_ner")
        & (audit_entities_df["text"].astype(str).str.len() <= 2)
    ]

    print(f"Total entities in Phase 4.3 output: {len(audit_entities_df):,}")
    print(
        f"Short (<=2 char) general_ner entities: {len(short_general):,} "
        f"({len(short_general) / max(len(audit_entities_df), 1):.1%} of all entities)"
    )
    print("\nMost common short fragments:")
    print(short_general["text"].value_counts().head(10))

    disaster_surface_forms = audit_entities_df.loc[
        audit_entities_df["label"] == "DISASTER_TYPE", "text"
    ].nunique()
    print(f"\nDistinct DISASTER_TYPE surface forms (entity-level): {disaster_surface_forms}")
else:
    print("phase4_3_entities.csv not found — skipping entity-level audit.")

if FOUND_PATHS["structured"] is not None:
    audit_structured_records = []
    with open(FOUND_PATHS["structured"], "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                audit_structured_records.append(json.loads(line))

    disaster_field_values = {
        r["disaster_type"] for r in audit_structured_records
        if isinstance(r.get("disaster_type"), str) and r["disaster_type"]
    }
    print(
        f"\nDistinct disaster_type STRING values in the structured output: "
        f"{len(disaster_field_values)}"
    )
    print(sorted(disaster_field_values)[:15], "...")
else:
    print("phase4_3_structured_incidents.jsonl not found — skipping structured-output audit.")

phase4_3_entities.csv not found — skipping entity-level audit.
phase4_3_structured_incidents.jsonl not found — skipping structured-output audit.


In [5]:
# ============================================================
# 5A. GAZETTEERS
# ============================================================

DISASTER_TYPES = {
    "flood": [
        "flood", "flooding", "flash flood", "flash flooding",
        "floodwaters", "flood water", "inundation"
    ],
    "earthquake": [
        "earthquake", "quake", "aftershock", "tremor"
    ],
    "fire": [
        "fire", "wildfire", "forest fire", "bushfire", "blaze"
    ],
    "hurricane": [
        "hurricane", "cyclone", "typhoon", "tropical storm"
    ],
    "tornado": [
        "tornado", "twister"
    ],
    "landslide": [
        "landslide", "mudslide", "rockslide"
    ],
    "tsunami": [
        "tsunami"
    ],
    "volcano": [
        "volcanic eruption", "volcano", "eruption"
    ],
    "storm": [
        "storm", "thunderstorm", "windstorm"
    ]
}

# Inverted lookup: any surface term (lowercased) -> canonical category.
DISASTER_TYPE_CANONICAL = {
    term.lower(): canonical
    for canonical, terms in DISASTER_TYPES.items()
    for term in terms
}

RESOURCE_TERMS = [
    "water", "drinking water", "food", "meals", "blankets",
    "medicine", "medicines", "medical supplies", "supplies",
    "clothes", "shelter", "tents", "fuel", "baby food",
    "bottled water", "first aid", "first aid kits",
    "blood", "oxygen", "generators"
]

INFRASTRUCTURE_TERMS = [
    "bridge", "road", "highway", "street", "building",
    "house", "homes", "hospital", "school", "airport",
    "railway", "railroad", "station", "dam", "power line",
    "power lines", "electricity grid", "pipeline", "port",
    "roadway", "overpass", "underpass"
]

RESCUE_TERMS = [
    "rescue", "rescued", "rescuing", "trapped", "stranded",
    "stuck", "evacuate", "evacuated", "evacuation",
    "missing", "search and rescue", "sos", "save us"
]

REQUEST_TERMS = [
    "need", "needs", "needed", "require", "requires",
    "required", "request", "requesting", "please send",
    "please provide", "looking for", "asking for",
    "urgently need", "urgent need", "help needed",
    "help us"
]

DISPLACEMENT_TERMS = [
    "displaced", "homeless", "evacuated", "evacuation",
    "forced to leave", "left their homes", "lost their homes",
    "without shelter", "shelter needed"
]

print("Gazetteers loaded.")
print("Disaster types:", len(DISASTER_TYPES), "| canonical surface terms:", len(DISASTER_TYPE_CANONICAL))

Gazetteers loaded.
Disaster types: 9 | canonical surface terms: 32


In [6]:
# ============================================================
# 5B. REGEX PATTERNS
# (casualty copula fix + displaced dedup fix from Phase 4.3 carried over)
# ============================================================

NUMBER_RE = re.compile(
    r"(?<!\w)(\d{1,6}(?:[,.]\d{3})*(?:\.\d+)?)(?!\w)",
    re.IGNORECASE
)

CASUALTY_RE = re.compile(
    r"(?P<number>\d{1,6})\s*"
    r"(?P<descriptor>people|persons|person|residents|victims|"
    r"children|men|women|families|workers)?\s*"
    r"(?:are|were|have\s+been|had\s+been)?\s*"
    r"(?P<status>killed|dead|died|deadly|injured|hurt|missing|"
    r"trapped|rescued|fatalities|casualties|victims)",
    re.IGNORECASE
)

REQUEST_RE = re.compile(
    r"(?P<context>"
    r"(?:urgently\s+)?(?:need|needs|needed|require|requires|required|"
    r"request|requesting|please\s+(?:send|provide)|help\s+(?:needed|us)|"
    r"looking\s+for|asking\s+for)"
    r")\s+"
    r"(?P<item>[^.!?;,\n]{2,80})",
    re.IGNORECASE
)

DISPLACED_RE = re.compile(
    r"(?P<text>"
    r"\d{0,6}\s*(?:people|families|residents)?\s*"
    r"(?:are\s+)?(?:displaced|homeless|evacuated)|"
    r"(?:people|families|residents)\s+(?:were|are)\s+"
    r"(?:forced\s+to\s+leave|displaced)|"
    r"(?:lost|have\s+lost)\s+(?:their\s+)?homes"
    r")",
    re.IGNORECASE
)

print("Regex patterns compiled.")

Regex patterns compiled.


In [7]:
# ============================================================
# 5C. UTILITY FUNCTIONS
# ============================================================

def clean_span(text):
    return re.sub(r"\s+", " ", str(text)).strip(" ,.;:!?")

def add_entity(entities, text, start, end, label, score=1.0, source="rule"):
    text = clean_span(text)
    if not text or start is None or end is None or end <= start:
        return

    entities.append({
        "text": text,
        "start": int(start),
        "end": int(end),
        "label": label,
        "score": float(score),
        "source": source
    })

def overlaps(a, b):
    return max(a["start"], b["start"]) < min(a["end"], b["end"])

def normalize_label(label):
    return {
        "PER": "PERSON",
        "ORG": "ORGANIZATION",
        "LOC": "LOCATION",
        "MISC": "MISC"
    }.get(label, label)

def normalize_entity_text(text, label):
    text = clean_span(text)

    if label in {"LOCATION", "ORGANIZATION", "PERSON", "INFRASTRUCTURE"}:
        return text.strip()

    return text.lower()

def deduplicate_entities(entities):
    best = {}

    for e in entities:
        key = (
            e["start"],
            e["end"],
            e["label"],
            normalize_entity_text(e["text"], e["label"])
        )

        if key not in best or e["score"] > best[key]["score"]:
            best[key] = e

    return list(best.values())

def is_probable_ner_fragment(span_text):
    '''
    Heuristic noise filter, tuned against the Phase 4.3 audit above.
    Drops 1-char spans outright; drops 2-char spans unless they're a
    fully-uppercase alphabetic code (US, TX, NZ, LA, ...).
    '''
    s = span_text.strip()
    if len(s) == 0:
        return True
    if len(s) == 1:
        return True
    if len(s) == 2:
        return not (s.isalpha() and s.isupper())
    return False

In [8]:
# ============================================================
# 5D. GENERAL NER EXTRACTION (with the new noise filter)
# ============================================================

def extract_general_ner(text, ner_pipeline):
    entities = []

    if not text.strip():
        return entities

    predictions = ner_pipeline(text)

    for p in predictions:
        score = float(p.get("score", 0.0))
        raw_label = p.get("entity_group", p.get("entity", ""))

        if score < GENERAL_NER_THRESHOLD:
            continue

        label = normalize_label(raw_label)

        if label not in {"LOCATION", "ORGANIZATION", "PERSON", "MISC"}:
            continue

        start = p.get("start")
        end = p.get("end")

        if start is None or end is None:
            word = clean_span(p.get("word", ""))
            start = text.find(word)
            end = start + len(word) if start >= 0 else None

        if start is not None and start >= 0 and end is not None:
            span_text = text[start:end]

            if is_probable_ner_fragment(span_text):
                continue

            add_entity(
                entities, span_text, start, end, label, score,
                source="general_ner"
            )

    return entities

In [9]:
# ============================================================
# 5E. DISASTER-SPECIFIC EXTRACTION LAYER
# ============================================================

def extract_disaster_type(text):
    entities = []
    lower = text.lower()

    for canonical, terms in DISASTER_TYPES.items():
        for term in terms:
            for match in re.finditer(
                rf"(?<!\w){re.escape(term)}(?!\w)", lower, flags=re.IGNORECASE
            ):
                add_entity(
                    entities, text[match.start():match.end()],
                    match.start(), match.end(), "DISASTER_TYPE",
                    RULE_HIGH, source="disaster_gazetteer"
                )

    return entities


def extract_numbers(text):
    entities = []
    for match in NUMBER_RE.finditer(text):
        add_entity(
            entities, match.group(1), match.start(1), match.end(1),
            "NUMBER", RULE_HIGH, source="number_regex"
        )
    return entities


def extract_casualties(text):
    entities = []

    for match in CASUALTY_RE.finditer(text):
        add_entity(
            entities, text[match.start():match.end()],
            match.start(), match.end(), "CASUALTY",
            RULE_HIGH, source="casualty_regex"
        )

    for match in re.finditer(
        r"(?<!\w)(\d{1,6})\s+"
        r"(fatalities|casualties|deaths|injuries|victims)(?!\w)",
        text, flags=re.IGNORECASE
    ):
        add_entity(
            entities, match.group(0), match.start(), match.end(),
            "CASUALTY", RULE_HIGH, source="casualty_regex"
        )

    return entities


def extract_requests(text):
    entities = []

    for match in REQUEST_RE.finditer(text):
        item = clean_span(match.group("item"))
        item_start = match.start("item")
        item_end = match.end("item")

        add_entity(
            entities, item, item_start, min(item_end, item_start + len(item)),
            "REQUEST", RULE_MEDIUM, source="request_regex"
        )

    return entities


def extract_resources(text):
    entities = []
    lower = text.lower()

    for term in sorted(RESOURCE_TERMS, key=len, reverse=True):
        for match in re.finditer(rf"(?<!\w){re.escape(term)}(?!\w)", lower):
            add_entity(
                entities, text[match.start():match.end()],
                match.start(), match.end(), "RESOURCE",
                RULE_MEDIUM, source="resource_gazetteer"
            )

    return entities


def extract_rescue(text):
    entities = []
    lower = text.lower()

    for term in sorted(RESCUE_TERMS, key=len, reverse=True):
        for match in re.finditer(rf"(?<!\w){re.escape(term)}(?!\w)", lower):
            add_entity(
                entities, text[match.start():match.end()],
                match.start(), match.end(), "RESCUE",
                RULE_MEDIUM, source="rescue_gazetteer"
            )

    return entities


def extract_displaced(text):
    entities = []

    for match in DISPLACED_RE.finditer(text):
        add_entity(
            entities, match.group(0), match.start(), match.end(),
            "DISPLACED", RULE_MEDIUM, source="displacement_regex"
        )

    lower = text.lower()
    for term in DISPLACEMENT_TERMS:
        for match in re.finditer(rf"(?<!\w){re.escape(term)}(?!\w)", lower):
            add_entity(
                entities, text[match.start():match.end()],
                match.start(), match.end(), "DISPLACED",
                RULE_LOW, source="displacement_gazetteer"
            )

    return entities


def extract_infrastructure(text):
    entities = []
    lower = text.lower()

    for term in sorted(INFRASTRUCTURE_TERMS, key=len, reverse=True):
        for match in re.finditer(rf"(?<!\w){re.escape(term)}(?!\w)", lower):
            add_entity(
                entities, text[match.start():match.end()],
                match.start(), match.end(), "INFRASTRUCTURE",
                RULE_MEDIUM, source="infrastructure_gazetteer"
            )

    return entities

In [10]:
# ============================================================
# 5F. CONFLICT RESOLUTION + FULL HYBRID EXTRACTION
# ============================================================

PARENT_ENTITY_PRIORITY = {
    "CASUALTY": 100,
    "DISPLACED": 90,
    "REQUEST": 80,
    "RESCUE": 80,
    "DISASTER_TYPE": 70,
    "INFRASTRUCTURE": 60,
    "RESOURCE": 50,
    "LOCATION": 40,
    "ORGANIZATION": 40,
    "PERSON": 40,
    "NUMBER": 10,
    "MISC": 1
}

def resolve_overlaps(entities):
    entities = deduplicate_entities(entities)

    ordered = sorted(
        entities,
        key=lambda e: (
            PARENT_ENTITY_PRIORITY.get(e["label"], 0),
            e["score"],
            e["end"] - e["start"]
        ),
        reverse=True
    )

    kept = []
    for candidate in ordered:
        conflicting = False
        for existing in kept:
            if overlaps(candidate, existing):
                if (
                    candidate["label"] == "NUMBER" and existing["label"] != "NUMBER"
                ) or (
                    existing["label"] == "NUMBER" and candidate["label"] != "NUMBER"
                ):
                    continue
                conflicting = True
                break
        if not conflicting:
            kept.append(candidate)

    return sorted(kept, key=lambda e: (e["start"], e["end"]))


def link_request_to_resources(text, request_entities, resource_entities):
    links = []
    for req in request_entities:
        nearby = [
            r for r in resource_entities
            if abs(r["start"] - req["start"]) <= 120
        ]
        for r in nearby:
            links.append({
                "request_text": req["text"],
                "resource_text": r["text"],
                "request_span": [req["start"], req["end"]],
                "resource_span": [r["start"], r["end"]]
            })
    return links


def extract_hybrid(text, ner_pipeline):
    text = str(text)
    entities = []

    entities.extend(extract_general_ner(text, ner_pipeline))
    entities.extend(extract_disaster_type(text))
    entities.extend(extract_numbers(text))
    entities.extend(extract_casualties(text))
    entities.extend(extract_requests(text))
    entities.extend(extract_resources(text))
    entities.extend(extract_rescue(text))
    entities.extend(extract_displaced(text))
    entities.extend(extract_infrastructure(text))

    entities = deduplicate_entities(entities)
    entities = resolve_overlaps(entities)

    for e in entities:
        e["text"] = text[e["start"]:e["end"]]
        e["normalized_text"] = normalize_entity_text(e["text"], e["label"])

    request_entities = [e for e in entities if e["label"] == "REQUEST"]
    resource_entities = [e for e in entities if e["label"] == "RESOURCE"]
    links = link_request_to_resources(text, request_entities, resource_entities)

    return {"text": text, "entities": entities, "request_resource_links": links}

In [11]:
# ============================================================
# 5G. STRUCTURED INCIDENT RECORD
# (disaster_type is now mapped through DISASTER_TYPE_CANONICAL)
# ============================================================

FIELD_MAP = {
    "LOCATION": "location",
    "CASUALTY": "casualties",
    "DISPLACED": "displaced",
    "REQUEST": "requests",
    "RESOURCE": "resources",
    "RESCUE": "rescue",
    "ORGANIZATION": "organizations",
    "PERSON": "persons",
    "NUMBER": "numbers",
    "INFRASTRUCTURE": "infrastructure"
}

def build_structured_record(result):
    text = result["text"]
    entities = result["entities"]

    record = {
        "text": text,
        "location": [], "casualties": [], "displaced": [], "requests": [],
        "resources": [], "rescue": [], "disaster_type": [], "organizations": [],
        "persons": [], "numbers": [], "infrastructure": [],
        "entities": entities,
        "request_resource_links": result["request_resource_links"]
    }

    for entity in entities:
        label = entity["label"]

        if label == "DISASTER_TYPE":
            canonical = DISASTER_TYPE_CANONICAL.get(
                entity["text"].lower(), entity["text"].lower()
            )
            if canonical not in record["disaster_type"]:
                record["disaster_type"].append(canonical)
            continue

        if label not in FIELD_MAP:
            continue

        field = FIELD_MAP[label]
        if entity["text"] not in record[field]:
            record[field].append(entity["text"])

    if len(record["disaster_type"]) == 1:
        record["disaster_type"] = record["disaster_type"][0]

    return record

## 6. Load the general NER model and re-run the audit sample

Confirms the fixes actually work before spending time on the full-corpus run.

In [12]:
# ============================================================
# 6A. LOAD GENERAL NER
# ============================================================

device = 0 if torch.cuda.is_available() else -1

general_ner = pipeline(
    "token-classification",
    model=NER_MODEL_NAME,
    tokenizer=NER_MODEL_NAME,
    aggregation_strategy="simple",
    device=device
)

print("General NER pipeline loaded.")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

General NER pipeline loaded.


In [13]:
# ============================================================
# 6B. RE-RUN ON THE SAME TWEETS PHASE 4.3 PROCESSED, COMPARE
# ============================================================

if audit_structured_records is not None:
    audit_texts = [r["text"] for r in audit_structured_records]

    new_results = []
    for text in tqdm(audit_texts, desc="Re-running fixed pipeline (audit sample)"):
        result = extract_hybrid(text, general_ner)
        new_results.append(build_structured_record(result))

    new_entity_counter = defaultdict(int)
    new_short_general = 0

    for r in new_results:
        for e in r["entities"]:
            new_entity_counter[e["label"]] += 1
            if e["source"] == "general_ner" and len(e["text"]) <= 2:
                new_short_general += 1

    new_disaster_values = {
        r["disaster_type"] for r in new_results
        if isinstance(r.get("disaster_type"), str) and r["disaster_type"]
    }

    print("BEFORE vs AFTER (same", len(audit_texts), "tweets):")
    print(f"  Short (<=2 char) general_ner entities : "
          f"{len(audit_entities_df[(audit_entities_df['source']=='general_ner') & (audit_entities_df['text'].astype(str).str.len()<=2)])} -> {new_short_general}")
    print(f"  Distinct disaster_type string values  : "
          f"{len({r['disaster_type'] for r in audit_structured_records if isinstance(r.get('disaster_type'), str) and r['disaster_type']})} -> {len(new_disaster_values)}")
    print("\nCanonical disaster_type values now in use:", sorted(new_disaster_values))
else:
    print("No Phase 4.3 structured output found — skipping before/after comparison.")
    print("(This is fine; the fixed pipeline below will still be used for the full run.)")

No Phase 4.3 structured output found — skipping before/after comparison.
(This is fine; the fixed pipeline below will still be used for the full run.)


In [14]:
# ============================================================
# 7A. PRE-FILL THE GOLD ANNOTATION TEMPLATE WITH MODEL PREDICTIONS
# ============================================================

def to_gold_format(entities):
    return [
        {"text": e["text"], "start": e["start"], "end": e["end"], "label": e["label"]}
        for e in entities
    ]

gold_template_df = None

if FOUND_PATHS["gold_template"] is not None:
    gold_template_df = pd.read_csv(FOUND_PATHS["gold_template"])

    prefilled_rows = []
    for _, row in tqdm(
        gold_template_df.iterrows(),
        total=len(gold_template_df),
        desc="Pre-filling gold template"
    ):
        result = extract_hybrid(str(row["text"]), general_ner)
        prefilled_rows.append({
            "annotation_id": row["annotation_id"],
            "text": row["text"],
            "entities_json": json.dumps(to_gold_format(result["entities"]), ensure_ascii=False)
        })

    prefilled_df = pd.DataFrame(prefilled_rows)
    prefilled_path = OUTPUT_DIR / "gold_annotation_prefilled.csv"
    prefilled_df.to_csv(prefilled_path, index=False)

    print("Pre-filled annotation draft saved to:", prefilled_path)
    print(
        "\nDownload this file, review/correct every row in a spreadsheet tool, "
        f"then upload the corrected version back as '{GOLD_FILE_NAME}'."
    )
    display(prefilled_df.head(5))
else:
    print("gold_annotation_template.csv not found — cannot pre-fill. Re-run Phase 4.3's template step first.")

gold_annotation_template.csv not found — cannot pre-fill. Re-run Phase 4.3's template step first.


In [15]:
# ============================================================
# 7B. LOAD THE (MANUALLY CORRECTED) GOLD FILE, IF IT EXISTS
# ============================================================

GOLD_CANDIDATES = [
    OUTPUT_DIR / GOLD_FILE_NAME,
    Path("/kaggle/working") / GOLD_FILE_NAME,
]
if FOUND_PATHS["gold_template"] is not None:
    GOLD_CANDIDATES.append(FOUND_PATHS["gold_template"].parent / GOLD_FILE_NAME)

def load_gold_annotations(candidates):
    for path in candidates:
        if path.exists():
            gold = pd.read_csv(path)
            required = {"annotation_id", "text", "entities_json"}
            missing = required - set(gold.columns)
            if missing:
                raise ValueError(f"Gold file {path} is missing columns: {sorted(missing)}")

            parsed = []
            for raw in gold["entities_json"].fillna("[]"):
                try:
                    entities = json.loads(raw)
                    parsed.append(entities if isinstance(entities, list) else [])
                except Exception:
                    parsed.append([])
            gold["gold_entities"] = parsed
            print("Loaded gold annotations from:", path)
            return gold

    print(
        "No gold_annotations.csv found yet in:",
        [str(c) for c in candidates],
        "\nComplete manual annotation (see Section 7A) first."
    )
    return None

gold_df = load_gold_annotations(GOLD_CANDIDATES)

if gold_df is not None:
    print("Gold rows:", len(gold_df))
    display(gold_df.head())

No gold_annotations.csv found yet in: ['/kaggle/working/phase4_4_5_outputs/gold_annotations.csv', '/kaggle/working/gold_annotations.csv'] 
Complete manual annotation (see Section 7A) first.


In [16]:
# ============================================================
# 7C. ENTITY-LEVEL EVALUATION (STRICT + RELAXED)
# ============================================================

def entity_key(entity):
    return (int(entity["start"]), int(entity["end"]), str(entity["label"]))

def score_table(tp, fp, fn):
    labels = sorted(set(tp) | set(fp) | set(fn))
    rows = []
    total_tp = total_fp = total_fn = 0

    for label in labels:
        t, f_p, f_n = tp[label], fp[label], fn[label]
        precision = t / (t + f_p) if (t + f_p) else 0.0
        recall = t / (t + f_n) if (t + f_n) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({"label": label, "TP": t, "FP": f_p, "FN": f_n,
                      "precision": precision, "recall": recall, "f1": f1})
        total_tp += t; total_fp += f_p; total_fn += f_n

    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0

    return pd.DataFrame(rows), {
        "micro_precision": micro_p, "micro_recall": micro_r, "micro_f1": micro_f1
    }


def evaluate_entity_predictions(pred_records, gold_df, mode="strict"):
    pred_by_text = defaultdict(list)
    for record in pred_records:
        pred_by_text[record["text"]].extend(record["entities"])

    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    evaluated = 0

    for _, row in gold_df.iterrows():
        text = str(row["text"])
        gold_entities = [e for e in row["gold_entities"] if all(k in e for k in ["start", "end", "label"])]
        pred_entities = [e for e in pred_by_text.get(text, []) if all(k in e for k in ["start", "end", "label"])]

        if mode == "strict":
            gold_keys = {entity_key(e) for e in gold_entities}
            pred_keys = {entity_key(e) for e in pred_entities}

            for key in pred_keys & gold_keys:
                tp[key[2]] += 1
            for key in pred_keys - gold_keys:
                fp[key[2]] += 1
            for key in gold_keys - pred_keys:
                fn[key[2]] += 1
        else:
            matched_gold = set()
            matched_pred = set()

            for gi, g in enumerate(gold_entities):
                for pi, p in enumerate(pred_entities):
                    if pi in matched_pred:
                        continue
                    if g["label"] == p["label"] and max(g["start"], p["start"]) < min(g["end"], p["end"]):
                        tp[g["label"]] += 1
                        matched_gold.add(gi)
                        matched_pred.add(pi)
                        break

            for gi, g in enumerate(gold_entities):
                if gi not in matched_gold:
                    fn[g["label"]] += 1
            for pi, p in enumerate(pred_entities):
                if pi not in matched_pred:
                    fp[p["label"]] += 1

        evaluated += 1

    report, summary = score_table(tp, fp, fn)
    summary["evaluated_tweets"] = evaluated
    return report, summary


if gold_df is not None:
    # Predictions for exactly the gold-annotated texts, using the fixed pipeline.
    gold_pred_records = []
    for text in tqdm(gold_df["text"].tolist(), desc="Predicting on gold sample"):
        result = extract_hybrid(str(text), general_ner)
        gold_pred_records.append(result)

    strict_report, strict_summary = evaluate_entity_predictions(gold_pred_records, gold_df, mode="strict")
    relaxed_report, relaxed_summary = evaluate_entity_predictions(gold_pred_records, gold_df, mode="relaxed")

    print("STRICT (exact span match):")
    print(json.dumps(strict_summary, indent=2))
    display(strict_report.sort_values("f1", ascending=False))

    print("\nRELAXED (any overlap, same label):")
    print(json.dumps(relaxed_summary, indent=2))
    display(relaxed_report.sort_values("f1", ascending=False))
else:
    print("Evaluation skipped until gold_annotations.csv is available.")

Evaluation skipped until gold_annotations.csv is available.


In [17]:
# ============================================================
# 7D. CATEGORIZED ERROR ANALYSIS
# ============================================================

def categorized_error_analysis(pred_records, gold_df):
    pred_by_text = defaultdict(list)
    for record in pred_records:
        pred_by_text[record["text"]].extend(record["entities"])

    rows = []

    for _, row in gold_df.iterrows():
        text = str(row["text"])
        gold_entities = [e for e in row["gold_entities"] if all(k in e for k in ["start", "end", "label"])]
        pred_entities = [e for e in pred_by_text.get(text, []) if all(k in e for k in ["start", "end", "label"])]

        matched_gold, matched_pred = set(), set()

        # exact matches first
        gold_keys = {entity_key(e): gi for gi, e in enumerate(gold_entities)}
        for pi, p in enumerate(pred_entities):
            k = entity_key(p)
            if k in gold_keys:
                matched_gold.add(gold_keys[k])
                matched_pred.add(pi)

        # then boundary-mismatch overlaps among what's left
        for gi, g in enumerate(gold_entities):
            if gi in matched_gold:
                continue
            for pi, p in enumerate(pred_entities):
                if pi in matched_pred:
                    continue
                if g["label"] == p["label"] and max(g["start"], p["start"]) < min(g["end"], p["end"]):
                    rows.append({
                        "error_type": "BOUNDARY_MISMATCH", "text": text, "label": g["label"],
                        "gold_span": g["text"], "pred_span": p["text"]
                    })
                    matched_gold.add(gi)
                    matched_pred.add(pi)
                    break

        for gi, g in enumerate(gold_entities):
            if gi not in matched_gold:
                rows.append({
                    "error_type": "FALSE_NEGATIVE", "text": text, "label": g["label"],
                    "gold_span": g["text"], "pred_span": ""
                })

        for pi, p in enumerate(pred_entities):
            if pi not in matched_pred:
                rows.append({
                    "error_type": "FALSE_POSITIVE", "text": text, "label": p["label"],
                    "gold_span": "", "pred_span": p["text"]
                })

    return pd.DataFrame(rows)


if gold_df is not None:
    errors_df = categorized_error_analysis(gold_pred_records, gold_df)
    print("Errors:", len(errors_df))
    print(errors_df["error_type"].value_counts())
    display(errors_df.head(30))

    errors_df.to_csv(OUTPUT_DIR / "phase4_4_error_analysis.csv", index=False)
else:
    errors_df = pd.DataFrame()
    print("Error analysis will run once gold_annotations.csv is available.")

Error analysis will run once gold_annotations.csv is available.


In [18]:
# ============================================================
# 8A. LOAD HUMAID (ROBUSTLY — SAME FIX AS PHASE 4.3)
# ============================================================

from datasets import load_dataset

def find_text_column(columns):
    preferred = ["text", "tweet", "tweet_text", "content", "post", "message", "tweetText", "clean_text"]
    lower_map = {str(c).lower(): c for c in columns}

    for name in preferred:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    candidates = [c for c in columns if any(k in str(c).lower() for k in ["text", "tweet", "content", "message", "post"])]
    if candidates:
        return candidates[0]

    raise ValueError(f"Could not identify a text column. Actual columns are: {list(columns)}")

# HumAID's dataset script declares its validation split as "dev" in metadata
# but actually generates "validation". Without verification_mode="no_checks"
# this raises ExpectedMoreSplitsError.
dataset = load_dataset(HF_DATASET_ID, verification_mode="no_checks")

available_splits = list(dataset.keys())
splits_to_process = FINAL_SPLITS if FINAL_SPLITS is not None else available_splits

missing = set(splits_to_process) - set(available_splits)
if missing:
    raise ValueError(f"Configured FINAL_SPLITS {sorted(missing)} not in dataset splits {available_splits}")

print("Available splits:", available_splits)
print("Splits selected for the final run:", splits_to_process)

TEXT_COLUMN = find_text_column(dataset[splits_to_process[0]].column_names)
print("Detected text column:", TEXT_COLUMN)

Generating train split:   0%|          | 0/53531 [00:00<?, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split:   0%|          | 0/15160 [00:00<?, ? examples/s]

Available splits: ['train', 'validation', 'test']
Splits selected for the final run: ['train', 'validation', 'test']
Detected text column: tweet_text


In [19]:
# ============================================================
# 8B. RUN THE FIXED HYBRID PIPELINE OVER EVERY SELECTED SPLIT
# ============================================================

final_records = []

for split_name in splits_to_process:
    split_df = dataset[split_name].to_pandas().copy()
    split_df[TEXT_COLUMN] = split_df[TEXT_COLUMN].fillna("").astype(str)
    split_df = split_df[split_df[TEXT_COLUMN].str.strip().ne("")].reset_index(drop=True)

    if ROW_LIMIT_PER_SPLIT is not None:
        split_df = split_df.head(ROW_LIMIT_PER_SPLIT)

    print(f"\nProcessing split '{split_name}': {len(split_df):,} rows")

    for text in tqdm(split_df[TEXT_COLUMN].tolist(), desc=f"Hybrid extraction ({split_name})"):
        result = extract_hybrid(text, general_ner)
        record = build_structured_record(result)
        record["split"] = split_name
        final_records.append(record)

print(f"\nTotal records processed across all splits: {len(final_records):,}")


Processing split 'train': 300 rows


Hybrid extraction (train):   0%|          | 0/300 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Processing split 'validation': 300 rows


Hybrid extraction (validation):   0%|          | 0/300 [00:00<?, ?it/s]


Processing split 'test': 300 rows


Hybrid extraction (test):   0%|          | 0/300 [00:00<?, ?it/s]


Total records processed across all splits: 900


In [20]:
# ============================================================
# 8C. SAVE THE FINAL DELIVERABLE
# ============================================================

final_entities_rows = []
for idx, record in enumerate(final_records):
    for e in record["entities"]:
        final_entities_rows.append({"row_id": idx, "split": record["split"], "text": record["text"], **e})

final_entities_df = pd.DataFrame(final_entities_rows)

final_entities_path = OUTPUT_DIR / "phase4_final_entities.csv"
final_structured_path = OUTPUT_DIR / "phase4_final_structured_incidents.jsonl"
final_summary_path = OUTPUT_DIR / "phase4_final_summary.json"

final_entities_df.to_csv(final_entities_path, index=False)

with open(final_structured_path, "w", encoding="utf-8") as f:
    for record in final_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

label_counts = defaultdict(int)
split_counts = defaultdict(int)
for record in final_records:
    split_counts[record["split"]] += 1
    for e in record["entities"]:
        label_counts[e["label"]] += 1

final_summary = {
    "total_tweets_processed": len(final_records),
    "tweets_with_entities": sum(len(r["entities"]) > 0 for r in final_records),
    "total_entities": sum(len(r["entities"]) for r in final_records),
    "tweets_per_split": dict(split_counts),
    "entities_per_label": dict(sorted(label_counts.items(), key=lambda x: -x[1])),
    "row_limit_per_split": ROW_LIMIT_PER_SPLIT,
}

if gold_df is not None:
    final_summary["gold_evaluation_strict"] = strict_summary
    final_summary["gold_evaluation_relaxed"] = relaxed_summary

with open(final_summary_path, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print("Saved:")
print(" -", final_entities_path)
print(" -", final_structured_path)
print(" -", final_summary_path)
print("\nSummary:")
print(json.dumps(final_summary, indent=2))

Saved:
 - /kaggle/working/phase4_4_5_outputs/phase4_final_entities.csv
 - /kaggle/working/phase4_4_5_outputs/phase4_final_structured_incidents.jsonl
 - /kaggle/working/phase4_4_5_outputs/phase4_final_summary.json

Summary:
{
  "total_tweets_processed": 900,
  "tweets_with_entities": 882,
  "total_entities": 3085,
  "tweets_per_split": {
    "train": 300,
    "validation": 300,
    "test": 300
  },
  "entities_per_label": {
    "LOCATION": 960,
    "DISASTER_TYPE": 630,
    "NUMBER": 545,
    "ORGANIZATION": 403,
    "MISC": 133,
    "CASUALTY": 123,
    "PERSON": 93,
    "REQUEST": 78,
    "RESOURCE": 43,
    "INFRASTRUCTURE": 30,
    "RESCUE": 29,
    "DISPLACED": 18
  },
  "row_limit_per_split": 300
}
